# AgentCacheBench: M6 Strong Baselines Execution

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshirsagarps/agent-cache-bench/blob/main/notebooks/M6_baselines.ipynb)

This notebook executes **Milestone 6 (M6 Strong Baselines)** of AgentCacheBench across Baselines B0–B4 on Google Colab GPU runtimes:
- **B0**: Cold Recomputation (No cache reuse)
- **B1**: Native Runtime Reuse (Radix Tree prefix cache)
- **B2**: Persistent Session State
- **B3**: Memory-Constrained LRU Reuse
- **B4**: Sub-chunk Segment Alignment Reuse

In [ ]:
# Step 1: Environment Setup
import os, sys, json, torch
if not os.path.exists('agentcachebench'):
    !git clone https://github.com/kshirsagarps/agent-cache-bench.git
    %cd agent-cache-bench

!pip install -q numpy scipy pandas jsonschema pyyaml matplotlib pillow

from agentcachebench.runner.colab_sync import get_colab_gpu_provenance
from agentcachebench.runner.engine import BenchmarkRunner
from agentcachebench.workloads.tool_use import generate_tool_use_trajectory
from agentcachebench.workloads.coding import generate_coding_trajectory
from agentcachebench.workloads.rag import generate_rag_trajectory
from agentcachebench.workloads.multi_agent import generate_multi_agent_trajectory

gpu_info = get_colab_gpu_provenance()
print(f"Colab Execution Host: {gpu_info}")

In [ ]:
# Step 2: Execute Baselines B0 - B4 Across Workloads W1 - W4
import os, sys, json
runner = BenchmarkRunner(output_dir="results/raw")

workload_generators = {
    "W1_tool_use": lambda seed: generate_tool_use_trajectory(num_steps=6, seed=seed),
    "W2_coding": lambda seed: generate_coding_trajectory(num_steps=6, seed=seed),
    "W3_rag": lambda seed: generate_rag_trajectory(num_steps=6, seed=seed),
    "W4_multi_agent": lambda seed: generate_multi_agent_trajectory(seed=seed),
}

exp_counter = 101
for w_name, gen_func in workload_generators.items():
    traj = gen_func(seed=exp_counter)
    runner.run_experiment(f"ACB_M6_{exp_counter}_B0_{w_name}", w_name, traj, "S0", "B0", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B1_{w_name}", w_name, traj, "S1", "B1", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B2_{w_name}", w_name, traj, "S2", "B2", {"enable_pause_decay": False})
    exp_counter += 1
    runner.run_experiment(f"ACB_M6_{exp_counter}_B3_{w_name}", w_name, traj, "S4", "B3", {"enable_pause_decay": True, "max_cache_blocks": 128})
    exp_counter += 1

print(f"M6 Baselines Execution Finished. Raw outputs saved to results/raw/")

In [ ]:
# Step 5: Mount Google Drive & Upload Results
import os, sys, shutil, glob

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    
    drive_dest = '/content/drive/MyDrive/AgentCacheBench_Results'
    os.makedirs(drive_dest, exist_ok=True)
    os.makedirs(os.path.join(drive_dest, 'raw'), exist_ok=True)
    
    copied_files = 0
    # Copy summary JSONs
    if os.path.exists('results'):
        for item in os.listdir('results'):
            src_path = os.path.join('results', item)
            if os.path.isfile(src_path) and item.endswith('.json'):
                shutil.copy(src_path, drive_dest)
                copied_files += 1
                print(f"Copied: {item} -> MyDrive/AgentCacheBench_Results/")
                
    # Copy raw trial JSONs
    if os.path.exists('results/raw'):
        for item in os.listdir('results/raw'):
            src_path = os.path.join('results/raw', item)
            if os.path.isfile(src_path) and item.endswith('.json'):
                shutil.copy(src_path, os.path.join(drive_dest, 'raw'))
                copied_files += 1
                print(f"Copied raw: {item} -> MyDrive/AgentCacheBench_Results/raw/")

    print(f"\n✅ SUCCESS: {copied_files} result files uploaded to Google Drive!")
    print(f"Folder Path: My Drive / AgentCacheBench_Results /")
except ImportError:
    print("Not running in Google Colab or google.colab module unavailable.")
except Exception as e:
    print(f"Drive Upload Exception: {e}")